# 00 · Preparación del dataset para el modelo causal

**Proyecto:** El Sol Que Más Rinde — Estimación del retorno marginal del gasto público contra la anemia infantil
**Eje:** Finanzas y Gobernabilidad · Bootcamp de IA, PUCP
**Notebook:** `notebooks/03_causal_model/00_prep_dataset.ipynb`

### Objetivo de este notebook

Tomar el panel distrital ya construido (`data/clean/merged/panel_distrital_clean.csv`) y dejarlo **listo para el modelo causal**, con una regla de diseño explícita:

> **Ningún distrito se elimina del archivo final.** Si un dato tiene un problema (nulo, muestra chica, gasto inconsistente, contexto territorial faltante en origen), se **corrige donde se puede** y se **declara con una bandera** donde no — nunca se borra la fila en silencio. Esto es una decisión deliberada: el proyecto necesita cobertura completa de los distritos del Perú, y cada decisión de limpieza debe quedar trazable, no escondida.

Este notebook **no entrena ningún modelo**. Es el paso 0 de la carpeta `03_causal_model`:

```
00_prep_dataset.ipynb   ← este notebook: deja Y, T, X, W listos, con banderas de calidad
01_causal_forest.ipynb  ← Double ML + Causal Forest, estima τ(X)
02_contrafractual.ipynb ← recalcula τ modificando una variable de X a la vez
```


## 1. Fuente de datos y qué trae ya resuelto

El archivo de entrada es `data/clean/merged/panel_distrital_clean.csv` — la versión ya corregida por el equipo (desfase de año de RENAMU, bug de `#¡NULO!`, exclusión de 2 ubigeos sin geometría en absoluto — `130112` y `180107`, un problema distinto y más profundo que no se resuelve en este notebook: les falta la geometría completa, no solo 3 columnas, y arreglarlo requeriría encontrar esos 2 distritos en otro shapefile del INEI. Queda como limitación declarada, no como algo que este notebook intente resolver).

Lo que este notebook sí resuelve, sección por sección, sin borrar ningún distrito:

- El gasto en soles absolutos → se convierte a per cápita y log.
- Los outliers administrativos de gasto (unidades ejecutoras centrales) → se **marcan**, no se eliminan.
- Los nulos de RENAMU → se imputan, con bandera de "esto no se sabía".
- Los 19 distritos sin altitud/superficie/densidad de origen → se completan con dato **investigado en fuentes públicas** (o promedio departamental como último recurso), con su nivel de confianza declarado.
- Las filas sin muestra suficiente de niños evaluados, o sin anemia registrada → se **marcan**, no se eliminan.


In [ ]:
# --- Imports y configuración ---
from pathlib import Path
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

def find_project_root(marker="requirements.txt"):
    path = Path.cwd()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"No se encontró '{marker}' subiendo desde {path}")

PROJECT_ROOT = find_project_root()
DATA = PROJECT_ROOT / "data"
INPUT_PATH = DATA / "clean" / "merged" / "panel_distrital_clean.csv"
OUTPUT_DIR = DATA / "clean" / "merged"

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"INPUT_PATH   = {INPUT_PATH}  (existe: {INPUT_PATH.exists()})")


## 2. Carga de datos

`ubigeo` siempre como `string`, para no perder el cero a la izquierda de los departamentos 01-09.


In [ ]:
df_raw = pd.read_csv(INPUT_PATH, dtype={"ubigeo": str})

print(f"Shape: {df_raw.shape}")
print(f"Distritos únicos (ubigeo): {df_raw['ubigeo'].nunique()}")
print(f"Rango de años: {df_raw['anio'].min()} - {df_raw['anio'].max()}")
df_raw.head(3)


## 3. Auditoría inicial

Confirmamos programáticamente lo que dice la documentación del proyecto, antes de tocar nada.


In [ ]:
duplicados = df_raw.duplicated(subset=["ubigeo", "anio"]).sum()
print(f"Filas duplicadas en la llave (ubigeo, anio): {duplicados}")
assert duplicados == 0

ubigeo_mal_formado = (~df_raw["ubigeo"].str.match(r"^\d{6}$")).sum()
print(f"Ubigeos que no son 6 dígitos: {ubigeo_mal_formado}")
assert ubigeo_mal_formado == 0

nulos = df_raw.isna().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)
print("\nColumnas con nulos:")
print(nulos)


## 4. Ventana temporal de análisis

`gasto_total`/`gasto_anemia_pan` (SIAF) están completos 2021-2025. RENAMU (`personal_total`, `programa_anemia`) cubre bien 2021-2023 y cae a cero en 2024 **y en 2025**. En vez de cortar el panel en el último año donde RENAMU todavía tiene algo de dato (como se hacía antes), la ventana se extiende hasta **2025** y el hueco completo de RENAMU en 2024-2025 se resuelve igual que cualquier otro nulo de RENAMU: con imputación + bandera de "dato faltante" (sección 6) — nunca botando el año completo.


In [ ]:
cols_cobertura = [
    "ninos_evaluados", "prevalencia_anemia",
    "gasto_total", "gasto_anemia_pan",
    "personal_total", "programa_anemia", "centro_salud_municipal",
]
cobertura = df_raw.groupby("anio")[cols_cobertura].apply(lambda d: d.notna().sum())
print("Filas no nulas por año y variable (de un máximo de ~1889 distritos/año):\n")
print(cobertura.to_string())


In [ ]:
ANIO_MIN, ANIO_MAX = 2021, 2025

panel = df_raw[df_raw["anio"].between(ANIO_MIN, ANIO_MAX)].copy()
print(f"Filas tras acotar a {ANIO_MIN}-{ANIO_MAX}: {len(panel)} ({panel['ubigeo'].nunique()} distritos — se conservan los 1889)")


## 5. El marco causal: roles de las variables (Y, T, X, W)

| Rol | Significado | Variables |
|---|---|---|
| **Y** — resultado | Lo que queremos explicar | `prevalencia_anemia` (+ `ninos_evaluados`, `ninos_con_anemia`, bandera `y_no_disponible`) |
| **T** — tratamiento | El gasto cuyo efecto marginal se quiere estimar | `gasto_total_percapita`, `gasto_anemia_pan_percapita` (y log), bandera `t_no_disponible` |
| **X** — contexto territorial (eje de heterogeneidad) | Lo que mide la visión por computadora / lo geoespacial | 16 columnas — ver sección 11 |
| **W** — controles de confusión | Gestión municipal, no territorial | `personal_total`, `programa_anemia`, `centro_salud_municipal`, `anio`, macro-regiones |

Y, además, un quinto grupo que no existía en la primera versión de este notebook: **banderas de calidad**, que documentan en vez de eliminar: `muestra_pequena`, `outlier_administrativo`, `y_no_disponible`, `t_no_disponible`, `altitude_fuente`, `superficie_fuente`, `pob_densidad_2020_fuente`, `geo_investigacion_confianza`.


## 6. Controles de gestión municipal (W): nulos e indicadores de dato faltante

Igual que antes: bandera `*_missing` por variable, `personal_total` imputado con la mediana del panel, `programa_anemia` y `centro_salud_municipal` imputados con 0 (asunción conservadora explícita: ante ausencia de reporte, no se asume que el distrito sí tiene el programa o el centro de salud). La diferencia con la versión anterior es que ahora esto también cubre 2024 y 2025 completos — antes esos años simplemente no entraban a la ventana.


In [ ]:
w_cols = ["personal_total", "programa_anemia", "centro_salud_municipal"]

print("Nulos antes de imputar (ventana 2021-2025):")
print(panel[w_cols].isna().sum())

for c in w_cols:
    panel[f"{c}_missing"] = panel[c].isna().astype(int)

mediana_personal = panel["personal_total"].median()
panel["personal_total"] = panel["personal_total"].fillna(mediana_personal)
panel["programa_anemia"] = panel["programa_anemia"].fillna(0)
panel["centro_salud_municipal"] = panel["centro_salud_municipal"].fillna(0)

print(f"\nMediana usada para personal_total: {mediana_personal:.1f}")
print("\nFilas marcadas como 'dato faltante' por variable:")
print(panel[[f"{c}_missing" for c in w_cols]].sum())


## 7. Gasto per cápita, y las filas donde el tratamiento no se puede calcular

`gasto_total_percapita = gasto_total / ninos_evaluados`. Cuando `ninos_evaluados` es nulo o cero, esa división no tiene resultado — no es un caso de "poca muestra", es un caso de "no hay denominador". Esas filas **se quedan en el panel** (con su Y, X y W intactos si los tienen), pero quedan marcadas con `t_no_disponible = 1`: el tratamiento no está disponible para esa fila específica, y así debe tratarla `01_causal_forest.ipynb` — no se inventa un gasto per cápita donde no se puede calcular.

Aparte, cuando `ninos_evaluados` es positivo pero muy chico (menos de 10), el per cápita sí se calcula, pero es ruidoso — se marca con `muestra_pequena = 1`, sin excluir la fila.


In [ ]:
panel["muestra_pequena"] = (
    (panel["ninos_evaluados"].notna()) & (panel["ninos_evaluados"] > 0) & (panel["ninos_evaluados"] < 10)
).astype(int)

denominador = panel["ninos_evaluados"].where(panel["ninos_evaluados"] > 0)  # evita división entre 0
panel["gasto_total_percapita"] = panel["gasto_total"] / denominador
panel["gasto_anemia_pan_percapita"] = panel["gasto_anemia_pan"] / denominador
panel["log_gasto_total_percapita"] = np.log1p(panel["gasto_total_percapita"])
panel["log_gasto_anemia_pan_percapita"] = np.log1p(panel["gasto_anemia_pan_percapita"])

panel["t_no_disponible"] = panel["gasto_total_percapita"].isna().astype(int)

print(f"Filas con muestra_pequena=1 (0 < niños evaluados < 10): {panel['muestra_pequena'].sum()}")
print(f"Filas con t_no_disponible=1 (sin denominador o sin gasto): {panel['t_no_disponible'].sum()}")
print(f"Distritos únicos en el panel: {panel['ubigeo'].nunique()}  (ninguno se perdió)")


## 8. El sesgo de la unidad ejecutora administrativa — se marca, no se elimina

Igual que antes: el `ubigeo` del gasto (SIAF) identifica la unidad ejecutora, no necesariamente el distrito de intervención real. Distritos como Lima Cercado, San Isidro, Miraflores, San Borja, Jesús María o Arequipa muestran un gasto per cápita órdenes de magnitud por encima del resto porque ahí están las sedes que administran presupuesto de un ámbito mucho más amplio.

**Cambio respecto a la versión anterior de este notebook:** esas filas ya no se separan a un archivo aparte. Se quedan en `causal_model_data.csv`, marcadas con `outlier_administrativo = 1`. El dato de gasto sigue siendo real — el problema es solo que no representa gasto local — así que la anemia (Y) y el contexto territorial (X) de esos distritos se conservan intactos y usables; es específicamente el tratamiento (T) el que no debería tomarse como confiable en esas filas al estimar el efecto causal.


In [ ]:
p99_gasto = panel.loc[panel["t_no_disponible"] == 0, "gasto_total_percapita"].quantile(0.99)
panel["outlier_administrativo"] = (panel["gasto_total_percapita"] > p99_gasto).fillna(False).astype(int)

n_outliers = panel["outlier_administrativo"].sum()
distritos_outliers = sorted(
    panel.loc[panel["outlier_administrativo"] == 1, "distrito"].dropna().unique()
)

print(f"Umbral p99 de gasto_total_percapita: S/ {p99_gasto:,.0f} por niño evaluado")
print(f"Filas marcadas como outlier administrativo: {n_outliers} de {len(panel)}")
print(f"Distritos afectados ({len(distritos_outliers)}): {distritos_outliers}")


## 9. Variable de resultado (Y): se marca, no se elimina

Antes se descartaban las filas sin `prevalencia_anemia`. Ahora se quedan, marcadas con `y_no_disponible = 1` — el distrito sigue apareciendo en el panel (con su gasto, su contexto territorial y sus controles), solo que sin resultado observado ese año específico.

Limitación que sigue sin resolverse aquí, y que debe declararse siempre que se use `prevalencia_anemia`: viene del registro administrativo de tamizajes (SIEN), no de un censo — refleja quién *llegó* a tamizarse. La corrección con ENDES es un modelo aparte, todavía no incorporado.


In [ ]:
panel["y_no_disponible"] = panel["prevalencia_anemia"].isna().astype(int)
print(f"Filas con y_no_disponible=1: {panel['y_no_disponible'].sum()}")
print(f"Distritos únicos: {panel['ubigeo'].nunique()}  (ninguno se perdió)")


## 10. Nota sobre `region` — no es costa/sierra/selva

`region` es una copia literal de `departamento` (26 valores, no 3 categorías naturales) — no se usa como control por esa razón. Se usan `macroregion_inei` y `macroregion_minsa` en su lugar. Construir la clasificación real de costa/sierra/selva queda como trabajo futuro.


In [ ]:
print("Valores únicos de 'region':", panel["region"].nunique(), "(debería ser ~3 si fuera costa/sierra/selva)")
print("Valores únicos de 'macroregion_inei':", sorted(panel["macroregion_inei"].dropna().unique()))


## 11. Los 19 distritos sin contexto territorial de origen — completados con dato investigado

Estos 19 distritos (mayoría selva/sierra remota, varios de creación reciente) llegan sin `altitude`, `superficie` ni `pob_densidad_2020` desde `geobase_distrital.gpkg` — pero **sí tienen** las otras 13 columnas de contexto territorial (esas se calculan por satélite, no dependen del censo), y sí tienen gasto, anemia y RENAMU.

En vez de dejarlos en blanco o excluirlos, se investigó cada uno en fuentes públicas (JNE/Infogob, Gobiernos Regionales, y en su defecto agregadores como deperu.com/distrito.pe cruzando al menos 2 fuentes cuando fue posible). Decisión tomada con la dueña del proyecto: **se usa el dato investigado con su nivel de confianza declarado** (alta/media/baja), y solo se cae al **promedio departamental** en los casos donde de verdad no se encontró ningún dato en ninguna fuente. Todo queda marcado en 3 columnas de "fuente" (`altitude_fuente`, `superficie_fuente`, `pob_densidad_2020_fuente`) — nunca se presenta un dato investigado o promediado como si fuera censal.

**Discrepancias entre fuentes, resueltas por promedio simple y declaradas aquí:** Ahuayro (población: 1,407 vs 1,047 → se usó 1,227), Oronccoy (altitud: 3,394 vs 3,719 msnm → se usó 3,557), Santa Lucía (población: 5,848 vs 7,090 → se usó 6,469).

`pob_densidad_2020` para estos 19 no es la densidad censal 2020 real (no existe) — es población investigada dividida entre la superficie final (investigada o promedio departamental), documentada con la fuente `derivado_investigacion`.


In [ ]:
GEO_INVESTIGADO = {
    # ubigeo: altitud_msnm, superficie_km2, poblacion, confianza (alta/media/baja)
    "030612": dict(altitude=2049,   superficie=29.55,   poblacion=1227, confianza="baja",
                    nota="deperu.com/distrito.pe; población promediada por discrepancia (1407 vs 1047)"),
    "050413": dict(altitude=3844,   superficie=94.01,   poblacion=230,  confianza="media",
                    nota="deperu.com y distrito.pe coinciden en población y superficie"),
    "050511": dict(altitude=3557,   superficie=None,    poblacion=984,  confianza="baja",
                    nota="altitud promediada por discrepancia (3394 vs 3719); superficie no encontrada en ninguna fuente"),
    "050512": dict(altitude=690,    superficie=38.75,   poblacion=2038, confianza="media",
                    nota="deperu.com y distrito.pe coinciden"),
    "050513": dict(altitude=667,    superficie=112.07,  poblacion=2294, confianza="media",
                    nota="deperu.com y distrito.pe coinciden"),
    "050514": dict(altitude=None,   superficie=99.71,   poblacion=1902, confianza="media",
                    nota="altitud no encontrada (riesgo de confundir con Ninabamba de Cajamarca, se descartó esa fuente)"),
    "050515": dict(altitude=2467,   superficie=96.041,  poblacion=1488, confianza="media",
                    nota="deperu.com y distrito.pe coinciden"),
    "080914": dict(altitude=303,    superficie=9507.84, poblacion=7427, confianza="alta",
                    nota="Infogob/JNE, fuente oficial"),
    "080915": dict(altitude=671,    superficie=2942.30, poblacion=5683, confianza="baja",
                    nota="fuente única no oficial (distrito.pe), no se pudo corroborar con fuente gubernamental"),
    "080916": dict(altitude=662,    superficie=None,    poblacion=2288, confianza="baja",
                    nota="fuente única no oficial; superficie no encontrada en ninguna fuente"),
    "080917": dict(altitude=656,    superficie=658.26,  poblacion=3666, confianza="media",
                    nota="deperu.com y distrito.pe"),
    "080918": dict(altitude=529,    superficie=309.96,  poblacion=6072, confianza="media",
                    nota="deperu.com"),
    "090723": dict(altitude=3538,   superficie=None,    poblacion=1411, confianza="baja",
                    nota="fuente única no oficial; superficie no encontrada"),
    "090724": dict(altitude=2575,   superficie=None,    poblacion=1493, confianza="baja",
                    nota="fuente única no oficial; superficie no encontrada"),
    "090725": dict(altitude=2684,   superficie=None,    poblacion=1648, confianza="baja",
                    nota="fuente única no oficial (verificado que corresponde a Huancavelica, no al homónimo de Cajamarca); superficie no encontrada"),
    "100609": dict(altitude=None,   superficie=323.55,  poblacion=6000, confianza="baja",
                    nota="altitud no encontrada; población aproximada sin año censal claro"),
    "221006": dict(altitude=507,    superficie=None,    poblacion=6469, confianza="baja",
                    nota="superficie no encontrada; población promediada por discrepancia (5848 vs 7090)"),
    "250306": dict(altitude=304,    superficie=575.39,  poblacion=4604, confianza="alta",
                    nota="Gobierno Regional de Ucayali (IDER), citando Censo INEI 2017"),
    "250307": dict(altitude=397,    superficie=None,    poblacion=3881, confianza="alta",
                    nota="Gobierno Regional de Ucayali (IDER), citando Censo INEI 2017; superficie no encontrada"),
}
print(f"Distritos con dato investigado: {len(GEO_INVESTIGADO)}")

dept_avg = panel.groupby("departamento")[["altitude", "superficie", "pob_densidad_2020"]].mean()

for col in ["altitude_fuente", "superficie_fuente", "pob_densidad_2020_fuente"]:
    panel[col] = "censo_origen"
panel["geo_investigacion_confianza"] = pd.Series([pd.NA] * len(panel), dtype="object")

for ubigeo, info in GEO_INVESTIGADO.items():
    mask = panel["ubigeo"] == ubigeo
    if mask.sum() == 0:
        print(f"⚠️  ubigeo {ubigeo} no está en el panel de esta ventana — revisar")
        continue
    depto = panel.loc[mask, "departamento"].iloc[0]

    if info["altitude"] is not None:
        panel.loc[mask, "altitude"] = info["altitude"]
        panel.loc[mask, "altitude_fuente"] = "investigacion_manual"
    else:
        panel.loc[mask, "altitude"] = dept_avg.loc[depto, "altitude"]
        panel.loc[mask, "altitude_fuente"] = "promedio_departamental"

    if info["superficie"] is not None:
        panel.loc[mask, "superficie"] = info["superficie"]
        panel.loc[mask, "superficie_fuente"] = "investigacion_manual"
    else:
        panel.loc[mask, "superficie"] = dept_avg.loc[depto, "superficie"]
        panel.loc[mask, "superficie_fuente"] = "promedio_departamental"

    superficie_final = panel.loc[mask, "superficie"].iloc[0]
    panel.loc[mask, "pob_densidad_2020"] = info["poblacion"] / superficie_final
    panel.loc[mask, "pob_densidad_2020_fuente"] = "derivado_investigacion"
    panel.loc[mask, "geo_investigacion_confianza"] = info["confianza"]

x_check = ["altitude", "superficie", "pob_densidad_2020"]
print(f"\nNulos restantes en altitude/superficie/pob_densidad_2020: {panel[x_check].isna().sum().sum()}")
print(f"Distritos únicos: {panel['ubigeo'].nunique()}  (los 1889 se conservan)")


## 12. Selección y organización final de columnas

Todas las columnas de contexto territorial (X) quedan sin nulos — ninguna fila se excluye por esto. Se agregan las banderas de calidad como un quinto bloque explícito.


In [ ]:
cols_llave = ["ubigeo", "anio"]
cols_identificacion = ["departamento", "provincia", "distrito"]

cols_Y = ["ninos_evaluados", "ninos_con_anemia", "prevalencia_anemia", "y_no_disponible"]

cols_T = [
    "gasto_total", "gasto_anemia_pan",
    "gasto_total_percapita", "gasto_anemia_pan_percapita",
    "log_gasto_total_percapita", "log_gasto_anemia_pan_percapita",
    "t_no_disponible",
]

cols_X = [
    "pct_cultivo", "pct_construido", "pct_desnudo", "pct_agua_visible",
    "n_edificios", "area_construida_m2", "confianza_media",
    "area_distrito_km2", "densidad_edificios_km2",
    "elevacion_media", "pendiente_media",
    "pct_agua_permanente", "pct_agua_estacional",
    "altitude", "superficie", "pob_densidad_2020",
]

cols_W = [
    "personal_total", "personal_total_missing",
    "programa_anemia", "programa_anemia_missing",
    "centro_salud_municipal", "centro_salud_municipal_missing",
    "anio", "macroregion_inei", "macroregion_minsa",
]

cols_banderas_calidad = [
    "muestra_pequena", "outlier_administrativo",
    "altitude_fuente", "superficie_fuente", "pob_densidad_2020_fuente",
    "geo_investigacion_confianza",
]

cols_finales = (
    cols_llave + cols_identificacion + cols_Y + cols_T + cols_X + cols_W + cols_banderas_calidad
)
cols_finales_unicas = list(dict.fromkeys(cols_finales))

causal_model_data = panel[cols_finales_unicas].copy()
print(f"Columnas finales ({len(causal_model_data.columns)}): {list(causal_model_data.columns)}")
print(f"Shape: {causal_model_data.shape}")


## 13. Validaciones finales

Se confirma que no hay duplicados en la llave, que X quedó sin nulos (es la única parte donde eso se exige, porque el causal forest necesita un valor de contexto para cada fila), y que Y y T solo tienen nulos donde su bandera correspondiente lo indica — nunca un nulo "silencioso".


In [ ]:
assert causal_model_data.duplicated(subset=["ubigeo", "anio"]).sum() == 0, "Duplicados en la llave"

nulos_x = causal_model_data[cols_X].isna().sum().sum()
assert nulos_x == 0, f"X no debería tener nulos, tiene {nulos_x}"

y_nulo_sin_bandera = causal_model_data["prevalencia_anemia"].isna() & (causal_model_data["y_no_disponible"] == 0)
assert y_nulo_sin_bandera.sum() == 0, "Hay nulos en Y sin su bandera correspondiente"

t_nulo_sin_bandera = causal_model_data["gasto_total_percapita"].isna() & (causal_model_data["t_no_disponible"] == 0)
assert t_nulo_sin_bandera.sum() == 0, "Hay nulos en T sin su bandera correspondiente"

print("✅ Sin duplicados en la llave (ubigeo, anio)")
print("✅ X sin nulos")
print("✅ Todo nulo en Y y T tiene su bandera correspondiente")
print(f"\nShape final: {causal_model_data.shape}")
print(f"Distritos únicos: {causal_model_data['ubigeo'].nunique()}  ——  el universo completo se conserva")
print(f"Años cubiertos: {sorted(causal_model_data['anio'].unique())}")
print()
print("Resumen de banderas de calidad:")
print(f"  muestra_pequena:        {causal_model_data['muestra_pequena'].sum()}")
print(f"  outlier_administrativo: {causal_model_data['outlier_administrativo'].sum()}")
print(f"  y_no_disponible:        {causal_model_data['y_no_disponible'].sum()}")
print(f"  t_no_disponible:        {causal_model_data['t_no_disponible'].sum()}")


## 14. Guardado

Un único archivo: **`causal_model_data.csv`**, en `data/clean/merged/`. Contiene el universo completo de distritos del Perú (2021-2025), con todas las banderas de calidad — nada se excluyó a un archivo aparte. Se guarda también un `.json` de metadata con la trazabilidad de cada decisión, para citar en la pantalla de metodología de la app sin tener que volver a correr el notebook.


In [ ]:
OUTPUT_PATH = OUTPUT_DIR / "causal_model_data.csv"
OUTPUT_METADATA_PATH = OUTPUT_DIR / "causal_model_data_metadata.json"

causal_model_data.to_csv(OUTPUT_PATH, index=False)

metadata = {
    "generado_en": datetime.now(timezone.utc).isoformat(),
    "fuente": str(INPUT_PATH.relative_to(PROJECT_ROOT)),
    "principio_de_diseno": "Ningún distrito se elimina del archivo final; los problemas de dato se corrigen o se declaran con banderas.",
    "ventana_temporal": {"anio_min": ANIO_MIN, "anio_max": ANIO_MAX},
    "shape_final": {"filas": int(causal_model_data.shape[0]), "columnas": int(causal_model_data.shape[1])},
    "distritos_unicos_final": int(causal_model_data["ubigeo"].nunique()),
    "anios_final": sorted(int(a) for a in causal_model_data["anio"].unique()),
    "banderas_de_calidad": {
        "muestra_pequena": int(causal_model_data["muestra_pequena"].sum()),
        "outlier_administrativo": int(causal_model_data["outlier_administrativo"].sum()),
        "y_no_disponible": int(causal_model_data["y_no_disponible"].sum()),
        "t_no_disponible": int(causal_model_data["t_no_disponible"].sum()),
        "umbral_outlier_administrativo_soles_percapita": float(p99_gasto),
    },
    "imputacion_renamu": {
        "personal_total": {"metodo": "mediana", "valor": float(mediana_personal)},
        "programa_anemia": {"metodo": "constante", "valor": 0},
        "centro_salud_municipal": {"metodo": "constante", "valor": 0},
    },
    "distritos_con_geo_investigado_manualmente": GEO_INVESTIGADO,
    "roles_de_variables": {
        "Y": cols_Y, "T": cols_T, "X": cols_X, "W": cols_W,
        "banderas_calidad": cols_banderas_calidad, "llave": cols_llave,
    },
    "limitaciones_declaradas": [
        "prevalencia_anemia viene de registro administrativo (SIEN), no de un censo; sesgo de selección todavía no corregido con ENDES.",
        "El ubigeo del gasto (SIAF) identifica la unidad ejecutora, no necesariamente el distrito de intervención real; los casos extremos quedan marcados en outlier_administrativo, pero el proxy sigue siendo imperfecto para el resto de distritos.",
        "gasto_anemia_pan solo cubre el Programa Articulado Nutricional (0001).",
        "region no es costa/sierra/selva (es una copia de departamento); no se usó como control.",
        "2 ubigeos (130112, 180107) siguen fuera del panel — no tienen geometría en absoluto en geobase_distrital.gpkg, un problema distinto (y más grande) al de los 19 distritos con geo investigado en este notebook.",
        "Los 19 distritos con altitude/superficie/pob_densidad_2020 investigados manualmente dependen, en su mayoría, de fuentes no oficiales (agregadores web) — el nivel de confianza de cada uno queda declarado en geo_investigacion_confianza, y no debe tratarse con la misma certeza que el dato censal original.",
        "pob_densidad_2020 de los 19 distritos investigados no es densidad censal 2020 real — es población investigada (de años variables, 2017-2024) dividida entre la superficie final.",
    ],
}

with open(OUTPUT_METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2, default=str)

print(f"Guardado: {OUTPUT_PATH.relative_to(PROJECT_ROOT)}  ({len(causal_model_data)} filas, {causal_model_data['ubigeo'].nunique()} distritos)")
print(f"Guardado: {OUTPUT_METADATA_PATH.relative_to(PROJECT_ROOT)}")


## 15. Resumen y siguiente paso

`causal_model_data.csv` contiene el universo completo de distritos del Perú (2021-2025) — nada se descartó. Las filas con problemas de dato quedan marcadas, no eliminadas: `muestra_pequena`, `outlier_administrativo`, `y_no_disponible`, `t_no_disponible`, y las columnas `_fuente` para los 19 distritos con geo investigado. Las cifras exactas de cada corrida quedan en `causal_model_data_metadata.json`.

**Siguiente notebook: `01_causal_forest.ipynb`.** Ahí, no antes, se decide qué hacer con cada bandera — por ejemplo, si las filas con `t_no_disponible=1` o `outlier_administrativo=1` se excluyen solo del entrenamiento del tratamiento, o si `muestra_pequena` se usa para ponderar en vez de excluir. La decisión de "usar o no" se toma en el modelo; este notebook solo garantiza que la información para tomarla esté completa y trazable.
